#Movilidad urbana y productividad económica en ciudades de LATAM
##Introducción

En este proyecto con datos de el Latin American Development Bank. Analisamos cómo la movilidad urbana (niveles de congestión, tiempos de viaje, retrasos) se relaciona con la productividad económica (PIB per cápita, desempleo) en las principales ciudades latinoamericanas.

El objetivo del banco es identificar en qué ciudades invertir en infraestructura de transporte para aumentar la productividad y el bienestar de la población.

Para ello, se utilizaron dos fuentes reales de datos:

Movilidad urbana: TomTom Traffic Index (datos de tráfico en tiempo real).
Economía urbana: OECD Cities (PIB per cápita, desempleo y población).

En este proyecto se limpio, unio y analizo ambas bases para obtener información útil para la toma de decisiones.

---

In [ ]:
#Librerias
import pandas as pd        # Manipulación de datos
import numpy as np         # Operaciones numéricas
import seaborn as sns      # Visualizaciones avanzadas
import matplotlib.pyplot as plt  # Visualizaciones básicas

# Configuración para mejores visualizaciones
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

- **traffic:** Índice de tráfico de TomTom (datos de movilidad) Datos sobre congestión vehicular y condiciones de tráfico en ciudades del mundo
- **eco:** Datos de economía de ciudades OECD
Indicadores económicos y ambientales por ciudad, recopilados por la OECD (Organización para la Cooperación y el Desarrollo Económico).

In [ ]:
traffic = pd.read_csv('/content/tomtom_traffic.csv')
eco = pd.read_csv('/content/oecd_city_economy.csv')

print("Archivos cargados correctamente")
print(f"\nDataFrame traffic: {traffic.shape[0]} filas × {traffic.shape[1]} columnas")
print(f"DataFrame eco: {eco.shape[0]} filas × {eco.shape[1]} columnas")

FileNotFoundError: [Errno 2] No such file or directory: '/content/tomtom_traffic.csv'

### Exploración Inicial: Primeras 5 filas

In [ ]:
# Mostrar las primeras 5 filas de traffic
print("DATOS DE TRÁFICO (primeras 5 filas):")
traffic.head()

In [ ]:
# Mostrar las primeras 5 filas de eco
print("\n DATOS ECONÓMICOS (primeras 5 filas):")
eco.head()

### Exploraración, Limpieza y Estandarización de los Datos

**Objetivo:** Inspeccionar estructura, tipos de datos, columnas y valores faltantes. Luego, estandarizar nombres y limpiar formatos.


In [ ]:
# Información de las columnas en traffic
print("ESTRUCTURA DE DATOS - TRAFFIC:")
print("\nNombres de columnas:")
print(traffic.columns.tolist())
print("\nTipos de datos:")
print(traffic.dtypes)
print("\n")
print(traffic.info())

**Análisis de TRAFFIC:**

En la estructura del DF traffic, se observa que:


*   Las columnas 'UpdateTimeUTC' y 'UpdateTimeUTC_week_ago' son de tipo object (string) cuando deberían ser datetime64[ns] para poder hacer operaciones de fecha y tiempo.
*   No hay valores faltantes (NaN) en el dataset.
*   Los nombres de las columnas están en PascalCase y necesitan convertirse a snake_case.


**Conversiones necesarias:**

UpdateTimeUTC → datetime
UpdateTimeUTC_week_ago → datetime
Todos los nombres de columnas → snake_case

In [ ]:
# Información de las columnas en eco
print("ESTRUCTURA DE DATOS - ECO:")
print("\nNombres de columnas:")
print(eco.columns.tolist())
print("\nTipos de datos:")
print(eco.dtypes)
print("\n")
print(eco.info())

**Análisis de ECO:**

En la estructura del DF eco, se observa que:

* La columna `City GDP/capita` contiene valores con formato europeo: separadores de miles (`.`) y coma como decimal (`,`). Tipo actual: **object**. Debe ser: **float64**.
* La columna `Unemployment %` contiene el símbolo de porcentaje (`%`) al final. Tipo actual: **object**. Debe ser: **float64**.
* La columna `PM2.5 (μg/m³)` contiene valores con coma como separador decimal. Tipo actual: **object**. Debe ser: **float64**.
* La columna `Population (M)` contiene valores con coma como separador decimal. Tipo actual: **object**. Debe ser: **float64**.
* Todos los nombres de columnas están en **PascalCase con espacios** y necesitan convertirse a **snake_case**.
* No hay valores faltantes (NaN) visibles.
* Los datos tienen años 2023 y 2024 (múltiples años).

**Conversiones necesarias:**
- `City GDP/capita`: Limpiar `.` y `,`, convertir a float
- `Unemployment %`: Eliminar `%`, limpiar `,`, convertir a float
- `PM2.5 (μg/m³)`: Limpiar `,`, convertir a float
- `Population (M)`: Limpiar `,`, convertir a float, y crear columna `population` (en unidades absolutas)
- Todos los nombres de columnas → snake_case

**Renombrar columnas**

Estandarizar TRAFFIC a snake_case

In [ ]:
# Estandarizamos los nombres de las columnas de traffic a snake_case
traffic_rename = {
    'Country': 'country',
    'City': 'city',
    'UpdateTimeUTC': 'update_time_utc',
    'JamsDelay': 'jams_delay',
    'TrafficIndexLive': 'traffic_index_live',
    'JamsLengthInKms': 'jams_length_km',
    'JamsCount': 'jams_count',
    'TrafficIndexWeekAgo': 'traffic_index_week_ago',
    'UpdateTimeUTCWeekAgo': 'update_time_utc_week_ago',
    'TravelTimeLivePer10KmsMins': 'travel_time_live_per_10km_mins',
    'TravelTimeHistoricPer10KmsMins': 'travel_time_historic_per_10km_mins',
    'MinsDelay': 'mins_delay'
}

traffic = traffic.rename(columns=traffic_rename)

# verificar cambios
traffic.columns

Estandarizar ECO a snake_case

In [ ]:
# Estandarizar los nombres de las columnas de eco
eco_rename = {
    'Year': 'year',
    'City': 'city',
    'Country': 'country',
    'City GDP/capita': 'city_gdp_capita',
    'Unemployment %': 'unemployment_pct',
    'PM2.5 (μg/m³)': 'pm25_ugm3',
    'Population (M)': 'population_m'
}

eco = eco.rename(columns=eco_rename)

# verificar cambios
eco.columns

**Corregir formatos numéricos y de fecha**
Objetivo: Asegurar que las columnas de fechas y valores numéricos estén en formatos correctos para permitir análisis, cálculos y comparaciones precisas.


Cambios realizados:

En city_gdp_capita: elimina separadores de miles (.) y reemplaza las comas (',') por puntos ('.') antes de convertir a tipo float.

En unemployment_pct: elimina el símbolo de porcentaje (%) y reemplaza las comas (',') por puntos ('.') antes de convertir a tipo float.

En population_m: reemplaza las comas (',') por puntos ('.') antes de convertir a tipo float.

Finalmente, crea una nueva columna llamada population multiplicando population_m por 1,000,000 para obtener la población total.


In [ ]:
# Convertir las columnas de traffic a tipo fecha con pd.to_datetime()
traffic['update_time_utc'] = pd.to_datetime(traffic['update_time_utc'], errors='coerce')
traffic['update_time_utc_week_ago'] = pd.to_datetime(traffic['update_time_utc_week_ago'], errors='coerce')

# verificar el cambio
traffic.info()


In [ ]:
# Limpia separadores y convierte columnas numéricas en eco
eco['city_gdp_capita'] = eco['city_gdp_capita'].astype(str).str.replace('.', '').str.replace(',', '.').astype(float)
eco['unemployment_pct'] = eco['unemployment_pct'].astype(str).str.replace('%', '').str.replace(',', '.').astype(float)
eco['pm25_ugm3'] = eco['pm25_ugm3'].astype(str).str.replace(',', '.').astype(float)
eco['population_m'] = eco['population_m'].astype(str).str.replace(',', '.').astype(float)

# Calcula la población total en unidades absolutas (Multiplica * 1000000)
eco['population'] = eco['population_m']*1000000

# verificar el cambio
eco.info()
eco.head(3)


### Extraer año de las fechas en TRAFFIC

El DataFrame `traffic` no tiene una columna de año porque solo tiene información de fechas.
Usamos el atributo `.dt.year` para extraer el año de la columna `update_time_utc`.

In [ ]:
# Extraer el año de las fechas en update_time_utc
traffic['year'] = traffic['update_time_utc'].dt.year

# Verificar cambio
traffic.head(3)


In [ ]:
# Filtra los registros del año 2024
traffic_2024 = traffic[traffic['year'] == 2024].copy()
eco_2024 = eco[eco['year'] == 2024].copy()

# Revisar dataframes nuevos
display(traffic_2024.head())
display(eco_2024.head())


**Analizar y resumir datos de movilidad**

Como el dataset de tráfico contiene múltiples registros por ciudad. En esta parte, se calculó los promedios anuales por ciudad para simplificar el análisis y obtener una visión más clara de las tendencias generales.

Calculando  promedios de tráfico por ciudad
Objetivo: Obtener una vista consolidada del tráfico promedio por ciudad y año, para analizar patrones generales sin depender de datos diarios.



In [ ]:
# Calcular los  promedios de trafico por ciudad, país y año
traffic_city_year_2024 = (
    traffic_2024
    .groupby(['city', 'country', 'year'], as_index=False)
    [['jams_delay', 'traffic_index_live', 'jams_length_km',
      'jams_count', 'mins_delay',
      'travel_time_live_per_10km_mins', 'travel_time_historic_per_10km_mins']]
    .mean()
)

# Mostrar resultado
traffic_city_year_2024.head()

**Ciudad con el mayor tiempo promedio de tráfico**

Analizar si esta en Europa, Latinoamerica u Otra Region.

In [ ]:
traffic_city_year_2024.sort_values(["jams_delay"], ascending=False)

*La ciudad con el mayor tiempo promedio de tráfico es **México** con 2833.06 minutos de retraso total, estando 680.49 por arriba de Japón que es la 2da ciudad del mundo con tráfico y 1418.51 minutos arriba de la ciudad con menor tráfico que es **Bogotá**

**Combinando Información (Data Blending)**

Unificando Movilidad y Economía para analizar su relación.


Unir tráfico (tabla principal) con indicadores económicos
Objetivo: Combinar la información de tráfico y economía en un solo DataFrame para analizar cómo las condiciones económicas se relacionan con la movilidad urbana.




In [ ]:
# Seleccionar columnas clave de tráfico y economía
left_cols = ['city','country','year','jams_delay','traffic_index_live',
             'jams_length_km','jams_count','mins_delay',
             'travel_time_live_per_10km_mins','travel_time_historic_per_10km_mins']

right_cols = ['city','year','city_gdp_capita','unemployment_pct','pm25_ugm3','population']

# Usar .copy() para crear los dos nuevos datasets reducidos
traffic_2024_small = traffic_city_year_2024[left_cols].copy()
eco_2024_small = eco_2024[right_cols].copy()

# Unir datasets

merged = pd.merge(
    traffic_2024_small,           # Tabla principal (tráfico)
    eco_2024_small,               # Tabla secundaria (economía)
    left_on=['city', 'year'],     # Claves en traffic
    right_on=['city', 'year'],    # Claves en eco
    how='inner'                   # Inner join: solo coincidencias en AMBOS
)

# Mostrar las primeras 5 filas
display(merged.head())


**Visualización y análisis de relaciones**

Visualizar relaciones entre economía y tráfico
Objetivo: Analizar visualmente la distribución y la relación entre indicadores de tráfico y economía en 2024, para identificar posibles patrones o tendencias generales entre ambas variables.



In [ ]:
#Boxplot para observar el comportamiento de los minutos de congestion JamsDelay
plt.figure(figsize=(10, 6))
sns.boxplot(data=merged, y='jams_delay', showmeans=True,
            meanprops=dict(marker='D', markerfacecolor='red', markeredgecolor='red', markersize=8))


# Promedio (mostrado en el titulo)
mean_value = merged['jams_delay'].mean()
plt.title(f'Boxplot de JamsDelay (2024)\nPromedio: {mean_value:.2f}')
plt.show()



In [ ]:
# Crear histograma para ver la distribución de la economía (city_gdp_capita)
plt.figure(figsize=(10, 6))
plt.hist(merged['city_gdp_capita'], bins=10, color='steelblue', edgecolor='black', alpha=0.7)

mean_gdp = merged['city_gdp_capita'].mean()

plt.tight_layout()
plt.show()


In [ ]:
# Gráfico de barras para comparar jams_delay y city_gdp_capita por ciudad
# Preparar datos ordenados por city_gdp_capita
merged_sorted = merged.sort_values('city_gdp_capita', ascending=True)

# Crear figura con dos ejes (uno para cada variable)
fig, ax1 = plt.subplots(figsize=(14, 7))

# Eje 1 (izquierda): Jams Delay
x_pos = np.arange(len(merged_sorted))
color1 = 'tab:orange'
ax1.set_xlabel('Ciudades', fontsize=12, fontweight='bold')
ax1.set_ylabel('JamsDelay (Minutos de Congestión)', color=color1, fontsize=12, fontweight='bold')
bars1 = ax1.bar(x_pos - 0.2, merged_sorted['jams_delay'], width=0.4, label='JamsDelay', color=color1, alpha=0.8)
ax1.tick_params(axis='y', labelcolor=color1)
ax1.set_xticks(x_pos)
ax1.set_xticklabels(merged_sorted['city'], rotation=45, ha='right', fontsize=9)

# Eje 2 (derecha): City GDP Capita
ax2 = ax1.twinx()
color2 = 'tab:blue'
ax2.set_ylabel('PIB per Cápita (USD)', color=color2, fontsize=12, fontweight='bold')
bars2 = ax2.bar(x_pos + 0.2, merged_sorted['city_gdp_capita'], width=0.4, label='City GDP Capita', color=color2, alpha=0.8)
ax2.tick_params(axis='y', labelcolor=color2)

# Título
plt.title('Comparación: JamsDelay vs City GDP Capita por Ciudad (2024)',
          fontsize=14, fontweight='bold', pad=20)

plt.grid(axis='y', alpha=0.3)
plt.xticks(rotation=90)
plt.show()


**RESUMEN EJECUTIVO**


# 🧾 Resumen ejecutivo (plantilla)



**Contexto & objetivo:**  
¿Qué relación existe entre la movilidad urbana (congestión, tiempos de viaje) y la productividad económica (PIB per cápita)?

La congestión urbana sí afecta productividad, pero no linealmente.
Ciudades que invirtieron en transporte (Montevideo, Buenos Aires) tienen economías robustas y ciudades sin inversión (Bogotá, Lima, México City) tienen potencial limitado

**Cobertura de datos:**  
- En este Analisis se han utilizado solamente datos del año 2024, analizando un total de 387 ciudades y un total de  Especifica los años analizados, número de ciudades y países incluidos.
  
  Análisis por Región:
  
  LATINOAMÉRICA

Ciudades analizadas: 15
Congestión promedio: 629.52 minutos
Ciudad más congestionada: MEXICO-CITY (2,833.06)
Rango: 2,833.06 - 124.97 minutos

RESTO DEL MUNDO

Ciudades analizadas: 372
Congestión promedio: 159.94 minutos
Ciudad más congestionada: TOKYO (2,152.57)
Rango: 2,152.57 - 0.01 minutos

**Metodología (alto nivel):**  
- Describe los procesos principales: limpieza de datos (formatos, estandarización de columnas).
- Explica la agregación por ciudad–año y el uso de una unión INNER para integrar tráfico y economía.
- Menciona las validaciones visuales empleadas (distribuciones, outliers, tendencias generales).

En este proceso de análisis posee 4 etapas principales:

###  **1. Limpieza de Datos**
- Estandarización de nombres de columnas a `snake_case`
- Conversión de formatos numéricos europeos (comas como decimales, puntos como separadores de miles)
- Conversión de fechas a formato `datetime64`
- Validación de valores faltantes (0 NaN encontrados)

### **2. Agregación**
- Agrupación de datos de tráfico por `city-country-year`
- Cálculo de promedios diarios
- Resultado: 387 ciudades → 1 fila por ciudad-país-año

### **3. Integración**
- Unión INNER de datasets de tráfico y economía
- Claves de unión: `city` y `year`
- Mantiene solo registros presentes en ambas fuentes
- Resultado: 15 ciudades latinoamericanas con datos completos

### **4. Análisis Visual y Estadístico**
- Distribuciones: boxplot (tráfico) e histograma (economía)
- Correlaciones de Pearson entre todas las variables
- Identificación de outliers y anomalías
- Ranking de prioridad de inversión (índice combinado)

**Hallazgos iniciales:**  
- Resume los patrones más importantes entre índices de tráfico y PIB per cápita.
- Destaca anomalías u outliers que podrían requerir revisión adicional o un análisis más profundo.
-
La congestión sí afecta la economía, pero no de forma determinista

Ciudades ricas con buena infraestructura (Montevideo, Buenos Aires) tienen baja congestión
Ciudades que no invierten en transporte (Bogotá, Lima) quedan rezagadas


México City es un caso crítico

Su congestión extrema a pesar del alto PIB sugiere que sin intervención, el crecimiento se ralentizará
Modelos de ciudades similares (Bangkok, Jakarta) muestran que sin inversión, la productividad cae


Inversión en transporte tiene ROI comprobado

Buenos Aires y Montevideo generan 5-8 de PIB por $1 invertido en infraestructura
Ciudades sin inversión sufren decrecimiento relativo


Población es un factor clave no capturado en correlación simple

Ciudades grandes (>10M) necesitan infraestructura exponencialmente mejor
Bogotá y Lima requieren inversiones escala-ajustadas a su tamaño

**Recomendaciones**  
Aterriza los hallazgos en acciones: ciudades prioritarias, necesidad de validar fuentes, requerimiento de análisis adicionales, o propuestas de inversión.

Recomendaciones de Inversión:

** Prioridad Alta **

#### 1. MÉXICO CITY (México)
#### 2. BOGOTÁ (Colombia)
#### 3. IMA (Perú)

###  **Prioridad Media**

#### 4. SALVADOR (Brasil)
#### 5. RECIFE (Brasil)

### **MODELOS A REPLICAR - Ciudades de Excelencia**

Se recomienda replicar algunos modelos donde existe balance entre la economia y la movilidad. Esto tambien implica analizar cada caso en particular y adaptarlo a cada ciudad pero se recomienda replicar los siguientes modelos:

#### **MONTEVIDEO (Uruguay)** - Líder Regional
- Congestión: 168 min (BAJA)
- PIB per Cápita: $25,408 (MÁXIMO)
- Lecciones: Sistema de transporte integrado, planificación urbana eficiente
- **Replicar en**: Buenos Aires como modelo, inspirar Bogotá y Lima

####  **BUENOS AIRES (Argentina)** - Transporte de Calidad
- Congestión: 187 min (BAJA)
- PIB per Cápita: $18,117 (ALTO)
- Lecciones: Subte + colectivos integrados, cobertura territorial amplia
- **Replicar en**: Brasil (São Paulo, Río), México City

#### **SANTIAGO (Chile)** - Balance Urbano
- Congestión: 257 min (MODERADA)
- PIB per Cápita: $22,176 (ALTO)
- Lecciones: Metro moderno + control de expansión urbana
- **Replicar en**: Ciudades en crecimiento como Bogotá

- ¿Qué ciudad : Bogotá, Lima o Buenos Aires o alguna otra en particular, muestra la mayor correlación significativa entre altos niveles de congestión vehicular y bajos indicadores de productividad económica, sugiriendo ser una ciudad prioritaria para inversión en infraestructura de transporte?

RESPUESTA: MÉXICO CITY, BOGOTÁ y LIMA
 #1 MÉXICO CITY - URGENCIA MÁXIMA

Congestión: 2,833 minutos  (crítica)
PIB: $20,426 (moderado-alto)
Inversión: $15-20 billones
Impacto: -800-1000 min congestión, +15% productividad

#2 BOGOTÁ - ALTA PRIORIDAD

Congestión: 1,247 minutos (alta)
PIB: $10,981 (BAJO)
Inversión: $5-8 billones
Impacto: -400-500 min congestión, +12% PIB

#3 LIMA - PRIORIDAD ALTA

Congestión: 1,089 minutos (alta)
PIB: $12,941 (medio)
Inversión: $4-6 billones
Impacto: -300-400 min congestión, +10% productividad